# Compare all KT runs
Lê os summaries salvos pelos notebooks e compara métricas dos 9 experimentos.


In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
import matplotlib.pyplot as plt
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/GuilhermeDesoler/ai-core.git'
REPO_DIR = Path('/content/ai-core')
if IN_COLAB:
    if not REPO_DIR.exists():
        get_ipython().system(f'git clone -b improve/high-impact-training {REPO_URL} /content/ai-core')
    get_ipython().run_line_magic('cd', '/content/ai-core')
    get_ipython().system('pip install -q -r requirements.txt')
else:
    REPO_DIR = Path.cwd()
summary_dir = REPO_DIR / 'artifacts' / 'colab_summaries'
print('Summary dir:', summary_dir)
print('Exists:', summary_dir.exists())


In [ ]:
rows=[]
for p in sorted(summary_dir.glob('*_metrics.json')):
    data=json.loads(p.read_text()) if p.exists() else {}
    name=p.stem.replace('_metrics','')
    model, task = name.split('_',1)
    rows.append({'name':name,'model':model.upper(),'task':task,'best_val_auc':data.get('best_val_auc'),'best_val_loss':data.get('best_val_loss'),'test_auc':data.get('test_auc'),'test_loss':data.get('test_loss'),'test_acc':data.get('test_acc')})
results_df=pd.DataFrame(rows).sort_values(['task','model'])
results_df


In [ ]:
for metric in ['test_auc','test_loss','test_acc']:
    pivot=results_df.pivot(index='task',columns='model',values=metric)
    display(pivot)
    fig=plt.figure(figsize=(10,5))
    pivot.plot(kind='bar', ax=plt.gca())
    plt.title(f'Comparison - {metric}')
    plt.ylabel(metric)
    plt.grid(alpha=0.3)
    plt.show()
